# `demo.ipynb` 深度解析与教学 

## 模块概述

本 Jupyter Notebook (`demo.ipynb`) 是一个交互式演示，旨在展示 `animegan2-pytorch` 项目的核心功能——将真实人脸图像转换为动漫风格（“Face2Paint”）。它整合了模型加载、人脸检测、面部关键点定位、图像对齐与裁剪，以及最终的风格化推理和结果展示。

**在整体项目中的定位和作用：**

1.  **功能演示**：直观展示项目的核心能力，让用户可以快速体验模型效果。
2.  **使用示例**：提供了如何加载预训练模型、准备输入数据、调用核心处理函数 (`face2paint`) 以及显示结果的完整代码示例。
3.  **高级图像处理流程**：集成了 dlib 库进行人脸检测和关键点定位，并实现了 FFHQ 数据集风格的面部对齐和裁剪，这通常是为了提高生成对抗网络（GAN）在人脸相关任务上的表现，确保输入给风格化模型的是标准化的、高质量的人脸区域。
4.  **用户友好**：通过在 Notebook 中组织代码和说明，降低了用户的使用门槛，方便用户在自己的环境中复现和修改。

**内部逻辑结构划分（按主要代码单元格）：**

1.  **说明单元格**：提供项目链接和运行指南。
2.  **模型加载单元格**：使用 `torch.hub.load` 从指定的 GitHub 仓库加载预训练的 `generator` 模型和 `face2paint` 便捷处理函数。
3.  **人脸检测与对齐工具单元格**：定义了三个核心函数：
    * `get_dlib_face_detector`: 获取 dlib 的人脸检测器和形状预测器（如果本地没有预测器模型文件 `shape_predictor_68_face_landmarks.dat`，会自动下载）。
    * `display_facial_landmarks`: 使用 matplotlib 可视化检测到的人脸关键点。
    * `align_and_crop_face`: 根据检测到的关键点，对人脸进行对齐和裁剪，输出标准尺寸的人脸图像，这借鉴了 FFHQ 数据集的处理方法。
4.  **图像处理与推理单元格**：加载示例图像（或允许用户上传），调用人脸检测与对齐函数处理图像，然后将处理后的人脸图像输入到 `face2paint` 函数进行风格化，并展示结果。

**依赖的外部库与模块：**

* `torch`: PyTorch 深度学习框架。
* `PIL.Image` (Pillow): 用于图像处理。
* `requests`: 用于从 URL 加载示例图像。
* `dlib`: 用于人脸检测和关键点定位。
* `numpy`: 用于数值计算，特别是处理图像和关键点数据。
* `matplotlib.pyplot`: 用于图像和关键点的可视化。
* `os`, `collections`, `typing`: Python 标准库，用于文件系统操作、数据结构和类型提示。
* `animegan2-pytorch:main` (via `torch.hub.load`): 项目自身在 GitHub 上的主分支，包含 `hubconf.py` 中定义的模型和函数入口。

更多关于 AnimeGANv2 的信息请访问: https://github.com/bryandlee/animegan2-pytorch

**开始使用**: 请按 `Ctrl+F9` 或通过菜单 "Runtime > Run All" (运行时 > 全部运行) 来执行所有单元格。

**GPU 加速**: 为了获得更快的推理速度，您可以使用 GPU。只需在菜单中选择 "Runtime > Change runtime type" (运行时 > 更改运行时类型)，然后在 "Hardware Acceleration" (硬件加速器) 下拉菜单中选择 "GPU"。

In [ ]:
#@title 1. 加载 Face2Paint 模型
import torch
from PIL import Image

print("步骤 1：加载 AnimeGANv2 模型...")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = torch.hub.load("bryandlee/animegan2-pytorch:main", "generator", device=device, progress=False).eval()
face2paint = torch.hub.load("bryandlee/animegan2-pytorch:main", "face2paint", device=device, progress=False, side_by_side=True)
print(f"模型加载成功，使用设备: {device}")

**代码单元格解析 ("加载 Face2Paint 模型")：**

此单元格负责加载核心的深度学习模型和相关的处理函数。

* `import torch`: 导入 PyTorch 库，这是模型运行的基础。
* `from PIL import Image`: 导入 Pillow 库中的 `Image` 模块，用于后续的图像打开和处理。
* `device = "cuda" if torch.cuda.is_available() else "cpu"`: 自动检测系统中是否有可用的 CUDA GPU。如果有，则将 `device` 设置为 `"cuda"`，否则设置为 `"cpu"`。这使得代码可以在不同硬件环境下运行。
* `model = torch.hub.load("bryandlee/animegan2-pytorch:main", "generator", device=device).eval()`:
    * 使用 `torch.hub.load` 从指定的 GitHub 仓库 (`bryandlee/animegan2-pytorch:main`) 加载名为 `"generator"` 的模型。这会查找仓库中 `hubconf.py` 文件内定义的 `generator` 函数并执行它。
    * `device=device`: 将加载的模型参数直接放到先前检测到的 `device`（CPU 或 GPU）上。
    * `.eval()`: 将模型设置为评估模式。这对于包含 Dropout 或 BatchNorm 等层的模型很重要，因为这些层在训练和推理时的行为不同。在评估模式下，Dropout 失效，BatchNorm 使用其学习到的统计量。
* `face2paint = torch.hub.load("bryandlee/animegan2-pytorch:main", "face2paint", device=device, side_by_side=True)`:
    * 同样使用 `torch.hub.load` 加载名为 `"face2paint"` 的函数。这个函数通常封装了完整的图像预处理、模型推理和后处理流程。
    * `device=device`: 确保此函数内部使用的模型和数据也在正确的设备上。
    * `side_by_side=True`: 这是一个传递给 `face2paint` 函数的参数，指示其在生成风格化图像时，将原始输入图像和风格化输出图像并排显示，方便对比。

**设计思路与框架特性：**

* **PyTorch Hub**: 利用 PyTorch Hub 功能，极大地简化了模型的获取和加载过程。用户无需手动下载模型文件和权重，`torch.hub.load` 会自动处理这些。
* **设备自适应**: 通过 `torch.cuda.is_available()` 实现设备自适应，增强了代码的可移植性。
* **评估模式**: `.eval()` 是 PyTorch 中模型推理前的标准步骤，确保了模型行为的正确性。
* **便捷函数封装**: `face2paint` 函数将复杂的处理流程封装起来，用户只需传入模型和图像即可得到结果，非常友好。

In [ ]:
#@title 2. 定义人脸检测与对齐函数
import os
import dlib
import collections
from typing import Union, List
import numpy as np
from PIL import Image
import PIL.ImageFile
import scipy.ndimage
import matplotlib.pyplot as plt

print("步骤 2：定义人脸检测与对齐的辅助函数...")

def get_dlib_face_detector(predictor_path: str = "shape_predictor_68_face_landmarks.dat"):
    if not os.path.isfile(predictor_path):
        print(f"关键点预测器 '{predictor_path}' 未找到，正在下载...")
        model_file = "shape_predictor_68_face_landmarks.dat.bz2"
        os.system(f"wget http://dlib.net/files/{model_file} -q")
        os.system(f"bzip2 -dk {model_file}")
        print("下载完成。")
    detector = dlib.get_frontal_face_detector()
    shape_predictor = dlib.shape_predictor(predictor_path)
    def detect_face_landmarks(img: Union[Image.Image, np.ndarray]) -> List[np.ndarray]:
        if isinstance(img, Image.Image):
            img = np.array(img)
        faces = []
        dets = detector(img, 1)
        for d in dets:
            shape = shape_predictor(img, d)
            faces.append(np.array([[v.x, v.y] for v in shape.parts()]))
        return faces
    return detect_face_landmarks

def display_facial_landmarks(img: Image.Image, landmarks: List[np.ndarray], fig_size=[10, 10]):
    plot_style = {'marker': 'o', 'markersize': 4, 'linestyle': '-', 'lw': 2}
    pred_type = collections.namedtuple('prediction_type', ['slice', 'color'])
    pred_types = {
        'face': pred_type(slice(0, 17), (0.682, 0.780, 0.909, 0.5)),
        'eyebrow1': pred_type(slice(17, 22), (1.0, 0.498, 0.055, 0.4)),
        'eyebrow2': pred_type(slice(22, 27), (1.0, 0.498, 0.055, 0.4)),
        'nose': pred_type(slice(27, 31), (0.345, 0.239, 0.443, 0.4)),
        'nostril': pred_type(slice(31, 36), (0.345, 0.239, 0.443, 0.4)),
        'eye1': pred_type(slice(36, 42), (0.596, 0.875, 0.541, 0.3)),
        'eye2': pred_type(slice(42, 48), (0.596, 0.875, 0.541, 0.3)),
        'lips': pred_type(slice(48, 60), (0.596, 0.875, 0.541, 0.3)),
        'teeth': pred_type(slice(60, 68), (0.596, 0.875, 0.541, 0.4))
    }
    fig = plt.figure(figsize=fig_size)
    ax = fig.add_subplot(1, 1, 1)
    ax.imshow(img)
    ax.axis('off')
    for face in landmarks:
        for pt_type in pred_types.values():
            ax.plot(face[pt_type.slice, 0], face[pt_type.slice, 1], color=pt_type.color, **plot_style)
    plt.show()

def align_and_crop_face(
    img: Image.Image,
    landmarks: np.ndarray,
    expand: float = 1.0,
    output_size: int = 1024,
    transform_size: int = 4096,
    enable_padding: bool = True
):
    lm = landmarks
    lm_eye_left      = lm[36 : 42]
    lm_eye_right     = lm[42 : 48]
    lm_mouth_outer   = lm[48 : 60]
    eye_left    = np.mean(lm_eye_left, axis=0)
    eye_right   = np.mean(lm_eye_right, axis=0)
    eye_avg     = (eye_left + eye_right) * 0.5
    eye_to_eye  = eye_right - eye_left
    mouth_left  = lm_mouth_outer[0]
    mouth_right = lm_mouth_outer[6]
    mouth_avg   = (mouth_left + mouth_right) * 0.5
    eye_to_mouth = mouth_avg - eye_avg
    x = eye_to_eye - np.flipud(eye_to_mouth) * [-1, 1]
    x /= np.hypot(*x)
    x *= max(np.hypot(*eye_to_eye) * 2.0, np.hypot(*eye_to_mouth) * 1.8)
    x *= expand
    y = np.flipud(x) * [-1, 1]
    c = eye_avg + eye_to_mouth * 0.1
    quad = np.stack([c - x - y, c - x + y, c + x + y, c + x - y])
    qsize = np.hypot(*x) * 2
    shrink = int(np.floor(qsize / output_size * 0.5))
    if shrink > 1:
        rsize = (int(np.rint(float(img.size[0]) / shrink)), int(np.rint(float(img.size[1]) / shrink)))
        img = img.resize(rsize, Image.Resampling.LANCZOS)
        quad /= shrink
        qsize /= shrink
    border = max(int(np.rint(qsize * 0.1)), 3)
    crop = (int(np.floor(min(quad[:,0]))), int(np.floor(min(quad[:,1]))), int(np.ceil(max(quad[:,0]))), int(np.ceil(max(quad[:,1]))))
    crop = (max(crop[0] - border, 0), max(crop[1] - border, 0), min(crop[2] + border, img.size[0]), min(crop[3] + border, img.size[1]))
    if crop[2] - crop[0] < img.size[0] or crop[3] - crop[1] < img.size[1]:
        img = img.crop(crop)
        quad -= crop[0:2]
    pad = (int(np.floor(min(quad[:,0]))), int(np.floor(min(quad[:,1]))), int(np.ceil(max(quad[:,0]))), int(np.ceil(max(quad[:,1]))))
    pad = (max(-pad[0] + border, 0), max(-pad[1] + border, 0), max(pad[2] - img.size[0] + border, 0), max(pad[3] - img.size[1] + border, 0))
    if enable_padding and max(pad) > border - 4:
        pad = np.maximum(pad, int(np.rint(qsize * 0.3)))
        img_np = np.pad(np.float32(img), ((pad[1], pad[3]), (pad[0], pad[2]), (0, 0)), 'reflect')
        h, w, _ = img_np.shape
        y, x, _ = np.ogrid[:h, :w, :1]
        mask = np.maximum(1.0 - np.minimum(np.float32(x) / pad[0], np.float32(w-1-x) / pad[2]), 1.0 - np.minimum(np.float32(y) / pad[1], np.float32(h-1-y) / pad[3]))
        blur = qsize * 0.02
        img_np += (scipy.ndimage.gaussian_filter(img_np, [blur, blur, 0]) - img_np) * np.clip(mask * 3.0 + 1.0, 0.0, 1.0)
        img_np += (np.median(img_np, axis=(0,1)) - img_np) * np.clip(mask, 0.0, 1.0)
        img = Image.fromarray(np.uint8(np.clip(np.rint(img_np), 0, 255)), 'RGB')
        quad += pad[:2]
    img = img.transform((transform_size, transform_size), Image.Transform.QUAD, (quad + 0.5).flatten(), Image.Resampling.BILINEAR)
    if output_size < transform_size:
        img = img.resize((output_size, output_size), Image.Resampling.LANCZOS)
    return img

print("辅助函数定义成功。")

**代码单元格解析 ("定义人脸检测与对齐函数")：**

这个单元格包含了一系列用于人脸检测、关键点定位和对齐裁剪的辅助函数。这些预处理步骤对于人脸风格化任务非常重要，因为它们可以提供一个标准化的、居中的人脸图像给后续的 GAN 模型，从而显著提高生成质量。

**1. `get_dlib_face_detector(predictor_path: str = "shape_predictor_68_face_landmarks.dat")` 函数：**
* **目的**: 初始化并返回一个 dlib 人脸关键点检测函数。
* **依赖下载**: 检查本地是否存在 dlib 的68点人脸关键点模型文件 (`shape_predictor_68_face_landmarks.dat`)。如果不存在，则通过 `os.system` 调用 `wget` 从 dlib.net 下载压缩包 (`.dat.bz2`)，然后使用 `bzip2` 解压。
    * *框架特性/习惯*: 这种在代码中自动下载依赖的方式在 Notebook 和 Colab 环境中很常见，以确保代码开箱即用。
* **检测器初始化**: 创建 `dlib.get_frontal_face_detector()` (用于检测人脸边界框) 和 `dlib.shape_predictor(predictor_path)` (用于在检测到的人脸内部定位68个关键点)。
* **返回内部函数 `detect_face_landmarks(img)`**: 
    * 这个内部函数接收 PIL 图像或 NumPy 数组作为输入。
    * 将输入图像转换为 NumPy 数组（如果需要）。
    * 使用 `detector(img)` 找到图像中的所有人脸。
    * 对每个人脸 `d`，使用 `shape_predictor(img, d)` 预测其68个关键点。
    * 将关键点坐标 (x, y) 存储为 NumPy 数组并添加到 `faces` 列表中。
    * 返回 `faces` 列表，其中每个元素是一个包含某张人脸68个关键点坐标的 NumPy 数组。

**2. `display_facial_landmarks(img: Image, landmarks: List[np.ndarray], fig_size=[15, 15])` 函数：**
* **目的**: 使用 matplotlib 可视化输入图像及检测到的面部关键点。
* **绘图样式**: 定义了关键点和连接线的样式（标记、大小、线型、线宽）。
* **特征分组**: `pred_types` 字典定义了不同面部特征（如下巴轮廓、眉毛、鼻子、眼睛、嘴唇等）对应的关键点索引范围 (`slice`) 和颜色 (`color`)。
    * *数据结构*: 使用 `collections.namedtuple` 创建了一个简单的数据结构 `prediction_type` 来组织这些信息。
* **绘图过程**: 创建 matplotlib 图形和子图，显示原始图像，并关闭坐标轴。然后遍历 `landmarks` 列表中的每张人脸，再遍历 `pred_types` 中的每个特征类型，提取对应索引范围的关键点，并在图像上绘制出来。

**3. `align_and_crop_face(img: Image.Image, landmarks: np.ndarray, ...)` 函数：**
* **目的**: 根据 FFHQ 数据集的对齐方法，对检测到的人脸进行旋转、缩放和平移变换，以获得一个标准化的、居中的、固定大小的人脸裁剪图像。
    * *背景知识*: FFHQ (Flickr-Faces-HQ) 是一个高质量的人脸图像数据集，其对齐方法被广泛用于人脸生成和编辑任务，能确保人脸姿态和比例的一致性。
* **关键点解析**: 从输入的68点 `landmarks` 数组中，提取各个面部组件（下巴、眉毛、鼻子、眼睛、嘴巴）的关键点子集。
* **辅助向量计算**: 计算眼睛中心点 (`eye_avg`)、两眼间向量 (`eye_to_eye`)、嘴巴中心点 (`mouth_avg`)、眼中心到嘴中心向量 (`eye_to_mouth`)。
* **定向裁剪框选择**: 基于上述向量，通过一系列几何运算（包括旋转和平移的模拟）计算出一个四边形 (`quad`)，这个四边形定义了人脸在原始图像中的理想裁剪区域。`qsize` 是这个四边形的大致边长。
* **图像缩小 (Shrink)**: 如果计算出的 `qsize` 远大于目标输出尺寸 `output_size`，为了保持图像质量和提高效率，会先将原始图像和 `quad` 坐标按比例缩小。
* **裁剪 (Crop)**: 根据 `quad` 的边界，并考虑一定的边界扩展 (`border`)，从（可能已缩小的）图像中裁剪出包含人脸的矩形区域。同时更新 `quad` 坐标以适应裁剪后的图像。
* **填充 (Pad)**: 如果裁剪后的人脸区域不足以覆盖变换所需的大小，或者为了在后续变换中引入一些背景，会进行填充。这里使用了 NumPy 的 `np.pad` 进行反射填充，并通过高斯模糊和平滑处理填充区域的边缘，使其与人脸区域融合得更自然。
* **变换 (Transform)**: 使用 `img.transform` 方法，将经过填充和裁剪的人脸区域进行透视变换（`PIL.Image.QUAD`），将其映射到一个 `transform_size` x `transform_size` 的正方形图像中。变换的源四边形是 `quad`。
* **最终缩放**: 如果 `output_size` 小于 `transform_size`，则将变换后的图像再缩放到最终的 `output_size`。
* 返回对齐并裁剪后的人脸 PIL.Image 对象。

**设计思路与框架特性：**

* **模块化工具**: 将复杂的人脸处理流程分解为独立的函数，提高了代码的可读性和复用性。
* **dlib 集成**: 有效利用了 dlib 强大的面部特征检测能力。
* **FFHQ 对齐标准**: 采用成熟的人脸对齐方法，有助于提升后续风格化模型的效果，因为 GANs 通常在对齐良好的人脸数据上训练效果更佳。
* **图像处理细节**: `align_and_crop_face` 函数中包含了许多精细的图像处理步骤（如反射填充、边缘模糊），这些都是为了获得更高质量的对齐人脸。

In [ ]:
#@title 3. 处理图像并获取结果
import requests
from io import BytesIO
from PIL import Image

img_url = "https://this-person-does-not-exist.com/img/avatar-gen1121c2567427214741399d863f63b22b.jpg"

print(f"步骤 3：处理来自 URL 的图像: {img_url}")
try:
    response = requests.get(img_url)
    response.raise_for_status()
    img = Image.open(BytesIO(response.content)).convert("RGB")
    face_detector = get_dlib_face_detector()
    landmarks = face_detector(img)

    if landmarks:
        print(f"检测到 {len(landmarks)} 张人脸。正在处理第一张...")
        face = align_and_crop_face(img, landmarks[0], output_size=512)
        print("人脸已对齐。正在生成动漫风格图像...")
        # 在 Jupyter 中，单元格的最后一个表达式会自动显示其内容。
        # 我们使用 display() 函数以确保在所有环境中都能显式地渲染图像。
        display(face2paint(model=model, img=face))
    else:
        print("\n错误：未在图像中检测到任何人脸。请尝试另一张图片。")
        display(img)

except requests.exceptions.RequestException as e:
    print(f"\n错误：无法下载图像。{e}")
except Exception as e:
    print(f"\n错误：处理过程中发生意外。{e}")

**代码单元格解析 ("处理图像并获取结果")：**

这个最终的单元格是整个流程的执行者，它将前面加载的模型和定义的函数串联起来，完成从数据输入到结果输出的完整任务。

* **图像获取**: 
    * 通过 `requests.get(img_url)` 从指定的URL获取图像的二进制数据。
    * `response.raise_for_status()` 是一个很好的实践，它会检查HTTP请求是否成功（状态码为200-299），如果请求失败（如404 Not Found），则会立即抛出异常。
    * `Image.open(BytesIO(response.content))` 这行代码展示了Pillow库的强大之处。`BytesIO` 在内存中创建了一个临时的二进制文件对象，使得`Image.open`可以直接从内存中读取图像数据，而无需先将图片保存到本地磁盘，整个过程更高效、简洁。
* **调用工具函数**:
    * `face_detector = get_dlib_face_detector()`: 调用我们在上一个单元格中定义的工厂函数，获取一个已经初始化好的、可直接使用的人脸检测器。
    * `landmarks = face_detector(img)`: 使用该检测器处理输入图像，返回一个包含所有检测到的人脸关键点列表。
* **鲁棒性检查 (核心改进)**:
    * `if landmarks:`: 这是一个至关重要的条件判断。`face_detector(img)` 在未找到人脸时会返回一个空列表 `[]`。在Python中，空列表的布尔值为 `False`。这个判断避免了在没有检测到人脸的情况下，后续代码因尝试访问 `landmarks[0]` 而导致的 `IndexError` 崩溃。这是一个健壮程序必备的防御性编程技巧。
    * 如果未检测到人脸，`else` 分支会打印一条清晰的错误信息，并显示原始图像，方便用户排查问题。
* **执行核心流程**:
    * `face = align_and_crop_face(img, landmarks[0], output_size=512)`: 如果检测到人脸，就取出第一张人脸的关键点（`landmarks[0]`），调用对齐和裁剪函数，生成一个 `512x512` 像素的标准人脸图像。
    * `display(face2paint(model=model, img=face))`: 这是执行链的最后一环。将对齐后的人脸图像 `face` 和已加载的 `model` 传递给 `face2paint` 函数，该函数执行GAN的推理，生成最终的动漫风格图像，并将其与原图并排拼接。`display()` 函数确保这个结果图像能被正确地渲染在Notebook的输出区域。
* **异常处理**:
    * `try...except...`: 整个逻辑被包裹在一个 `try...except` 块中，用于捕获可能发生的各种异常，如网络请求失败 (`requests.exceptions.RequestException`) 或其他未知错误 (`Exception`)，并打印出友好的错误提示，而不是让程序以不友好的Traceback信息终止。